**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 20: LoRA vs Full Fine-tuning 이론 비교

## 🎯 LoRA vs Full Fine-tuning 비교

이번 세션에서는 **LoRA**와 **Full Fine-tuning(FFT)**을 이론적, 실험적으로 비교합니다.

### 두 방법의 핵심 차이

| 구분 | LoRA | Full Fine-tuning |
|------|------|------------------|
| 학습 파라미터 | 전체의 ~1% | 전체 100% |
| 메모리 사용 | 낮음 (4bit + LoRA) | 높음 (FP16/FP32) |
| 학습 속도 | 빠름 | 느림 |
| 저장 크기 | ~수십 MB (어댑터만) | ~수 GB (전체 모델) |
| 성능 | 약간 낮을 수 있음 | 최고 성능 가능 |
| RTX 4060 가능 모델 | ~7B (QLoRA) | ~1.5B |

### 학습 목표

- ✅ LoRA와 FFT의 파라미터 수 차이를 직접 확인
- ✅ 메모리 사용량 비교 (GPU 모니터링)
- ✅ 학습 속도 비교
- ✅ 저장 크기 비교
- ✅ 상황별 선택 가이드 이해

## 1️⃣ 파라미터 수 비교 (전체 vs LoRA 학습 파라미터)

먼저 Qwen2.5-1.5B 모델의 전체 파라미터 수와 LoRA가 학습하는 파라미터 수를 비교합니다.

In [ ]:
# 필수 라이브러리
import torch
import gc
import os
import time
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 임포트 완료")

In [ ]:
# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

def get_gpu_memory_gb():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**3
    return 0

print_gpu_memory("시작")

In [ ]:
# Qwen2.5-1.5B 파라미터 분석
MODEL_NAME = "Qwen/Qwen2.5-1.5B"

print("📊 Qwen2.5-1.5B 모델 파라미터 분석")
print("="*60)

# 1.5B 모델 구조 정보 (문서 기반)
model_info = {
    "총 파라미터": 1_500_000_000,
    "hidden_size": 1536,
    "num_layers": 28,
    "num_attention_heads": 12,
    "intermediate_size": 8960,
}

# LoRA 파라미터 계산
r = 16
hidden_size = model_info["hidden_size"]
num_layers = model_info["num_layers"]

# 어텐션 레이어: q, k, v, o_proj (4개)
attn_lora_params = 2 * r * hidden_size * 4 * num_layers

# FFN 레이어: gate, up, down_proj (3개) - intermediate_size 사용
intermediate_size = model_info["intermediate_size"]
ffn_lora_gate_up = 2 * r * intermediate_size * 2 * num_layers  # gate, up
ffn_lora_down = 2 * r * hidden_size * 1 * num_layers  # down

# 전체 LoRA 파라미터
attn_only_lora = attn_lora_params
full_lora = attn_lora_params + ffn_lora_gate_up + ffn_lora_down

total_params = model_info["총 파라미터"]

print(f"\n🔹 전체 모델 파라미터: {total_params:,} ({total_params/1e9:.1f}B)")
print(f"\n🔹 LoRA (어텐션만, r={r}):")
print(f"   학습 파라미터: {attn_only_lora:,} ({attn_only_lora/total_params*100:.3f}%)")
print(f"\n🔹 LoRA (어텐션+FFN, r={r}):")
print(f"   학습 파라미터: {full_lora:,} ({full_lora/total_params*100:.3f}%)")
print(f"\n🔹 Full Fine-tuning:")
print(f"   학습 파라미터: {total_params:,} (100.000%)")
print(f"\n📌 LoRA는 전체의 약 {full_lora/total_params*100:.2f}%만 학습! ({total_params//full_lora}배 적음)")

## 2️⃣ LoRA 실습: 모델 로드 → LoRA 적용 → 파라미터 확인

In [ ]:
# 공통 데이터 준비 (간단한 텍스트 데이터)
sample_texts = [
    "인공지능은 인간의 지능을 모방하여 학습, 추론, 판단 등의 작업을 수행하는 시스템입니다.",
    "머신러닝은 데이터로부터 패턴을 학습하는 알고리즘을 연구하는 분야입니다.",
    "딥러닝은 인공 신경망을 여러 층으로 쌓아 복잡한 패턴을 학습합니다.",
    "트랜스포머는 셀프 어텐션 메커니즘을 핵심으로 하는 아키텍처입니다.",
    "LoRA는 적은 파라미터만 학습하면서도 좋은 성능을 달성할 수 있습니다.",
] * 10  # 50개로 복제

print(f"✅ 샘플 데이터 준비 완료: {len(sample_texts)}개")

In [ ]:
# LoRA 모델 로드 (4bit 양자화)
print("="*60)
print("📊 LoRA 모델 로드")
print("="*60)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

torch.cuda.empty_cache()
gc.collect()
mem_before_lora = get_gpu_memory_gb()

lora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
lora_model.gradient_checkpointing_enable()

mem_after_load = get_gpu_memory_gb()
print(f"📌 4bit 모델 로드 GPU 메모리: {mem_after_load:.1f}GB")

In [ ]:
# LoRA 어댑터 적용
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()

mem_after_lora = get_gpu_memory_gb()
print(f"\n📌 LoRA 적용 후 GPU 메모리: {mem_after_lora:.1f}GB")
print(f"📌 LoRA 추가 메모리: {mem_after_lora - mem_after_load:.2f}GB")

In [ ]:
# LoRA 모델의 레이어 구조 확인
print("📊 LoRA 모델 레이어 구조 (첫 번째 Transformer 블록)")
print("="*60)
for name, param in lora_model.named_parameters():
    if "layers.0." in name:
        status = "🟢 학습" if param.requires_grad else "🔴 동결"
        print(f"  {status} {name}: {param.shape}, {param.dtype}")

## 3️⃣ FFT 실습: 모델 로드 → 전체 파라미터 학습 설정

⚠️ RTX 4060에서 FFT는 1.5B까지만 가능합니다. FP16으로 로드합니다.

In [ ]:
# LoRA 모델 메모리 해제
lora_mem_usage = get_gpu_memory_gb()
del lora_model
gc.collect()
torch.cuda.empty_cache()
print("✅ LoRA 모델 메모리 해제")
print_gpu_memory("LoRA 해제 후")

In [ ]:
# FFT 모델 로드 (FP16 - 양자화 없음)
print("="*60)
print("📊 FFT 모델 로드 (FP16)")
print("="*60)

mem_before_fft = get_gpu_memory_gb()

fft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

mem_after_fft = get_gpu_memory_gb()
print(f"📌 FP16 모델 로드 GPU 메모리: {mem_after_fft:.1f}GB")

In [ ]:
# FFT: 전체 파라미터 학습 가능으로 설정
trainable_params = 0
all_params = 0
for name, param in fft_model.named_parameters():
    all_params += param.numel()
    param.requires_grad = True  # 전체 파라미터 학습
    trainable_params += param.numel()

print(f"📊 FFT 파라미터 정보")
print(f"📌 전체 파라미터: {all_params:,}")
print(f"📌 학습 파라미터: {trainable_params:,}")
print(f"📌 학습 비율: {trainable_params/all_params*100:.1f}%")
print_gpu_memory("FFT 설정 후")

In [ ]:
# FFT 모델 레이어 구조 확인
print("📊 FFT 모델 레이어 구조 (첫 번째 Transformer 블록)")
print("="*60)
for name, param in fft_model.named_parameters():
    if "layers.0." in name:
        status = "🟢 학습" if param.requires_grad else "🔴 동결"
        print(f"  {status} {name}: {param.shape}, {param.dtype}")

## 4️⃣ 메모리 사용량 비교 (GPU 모니터링)

In [ ]:
# FFT 메모리 해제
fft_mem_usage = get_gpu_memory_gb()
del fft_model
gc.collect()
torch.cuda.empty_cache()

# 메모리 비교 결과
print("="*60)
print("📊 GPU 메모리 사용량 비교")
print("="*60)
print(f"\n🔹 LoRA (4bit + LoRA):  ~{lora_mem_usage:.1f}GB")
print(f"🔹 FFT (FP16):          ~{fft_mem_usage:.1f}GB")
print(f"\n📌 FFT는 LoRA 대비 약 {fft_mem_usage/max(lora_mem_usage, 0.1):.1f}배 메모리 사용")
print(f"📌 RTX 4060 (8GB)에서:")
print(f"   - LoRA: ✅ 여유 있음 (학습 시 ~{lora_mem_usage+1:.1f}GB 예상)")
print(f"   - FFT:  ⚠️ 빠듯함 (학습 시 ~{fft_mem_usage+2:.1f}GB 예상)")

## 5️⃣ 학습 속도 비교 (동일 데이터, 동일 에포크)

동일한 데이터와 설정으로 LoRA와 FFT의 학습 속도를 비교합니다.

In [ ]:
# 공통 데이터셋 준비
def prepare_dataset(tokenizer, texts, max_length=512):
    """텍스트를 토큰화하여 데이터셋 생성"""
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return Dataset.from_dict(tokenized)

dataset = prepare_dataset(tokenizer, sample_texts, max_length=256)
print(f"✅ 데이터셋 준비 완료: {len(dataset)}개 샘플")

In [ ]:
# LoRA 학습 시간 측정
print("="*60)
print("⏱️ LoRA 학습 시간 측정")
print("="*60)

# LoRA 모델 재로드
lora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
lora_model.gradient_checkpointing_enable()
lora_model = get_peft_model(lora_model, lora_config)

lora_training_args = TrainingArguments(
    output_dir="./output/lora_speed_test",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    fp16=True,
    logging_steps=5,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
lora_trainer = Trainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

print("🚀 LoRA 학습 시작...")
lora_start = time.time()
lora_result = lora_trainer.train()
lora_time = time.time() - lora_start
lora_loss = lora_result.training_loss

print(f"\n✅ LoRA 학습 완료")
print(f"📌 소요 시간: {lora_time:.1f}초")
print(f"📌 Training Loss: {lora_loss:.4f}")
print_gpu_memory("LoRA 학습 후")

# 메모리 해제
lora_peak_mem = get_gpu_memory_gb()
del lora_model, lora_trainer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# FFT 학습 시간 측정
print("="*60)
print("⏱️ FFT 학습 시간 측정")
print("="*60)

# FFT 모델 로드 (FP16)
fft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
fft_model.gradient_checkpointing_enable()

fft_training_args = TrainingArguments(
    output_dir="./output/fft_speed_test",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    fp16=True,
    logging_steps=5,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
)

fft_trainer = Trainer(
    model=fft_model,
    args=fft_training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

print("🚀 FFT 학습 시작...")
fft_start = time.time()
fft_result = fft_trainer.train()
fft_time = time.time() - fft_start
fft_loss = fft_result.training_loss

print(f"\n✅ FFT 학습 완료")
print(f"📌 소요 시간: {fft_time:.1f}초")
print(f"📌 Training Loss: {fft_loss:.4f}")
print_gpu_memory("FFT 학습 후")

fft_peak_mem = get_gpu_memory_gb()

## 6️⃣ 저장 크기 비교

In [ ]:
# 모델 저장 및 크기 비교
import shutil

# FFT 모델 저장
fft_save_path = "./output/fft_speed_test/saved_model"
print("⏳ FFT 모델 저장 중...")
fft_model.save_pretrained(fft_save_path)

# FFT 저장 크기 계산
fft_size = 0
for root, dirs, files in os.walk(fft_save_path):
    for f in files:
        fft_size += os.path.getsize(os.path.join(root, f))

# 메모리 해제
del fft_model, fft_trainer
gc.collect()
torch.cuda.empty_cache()

# LoRA 모델 저장 (재로드 필요)
print("⏳ LoRA 어댑터 저장을 위해 모델 재로드...")
lora_model_2 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
lora_model_2 = get_peft_model(lora_model_2, lora_config)

lora_save_path = "./output/lora_speed_test/lora_adapter"
lora_model_2.save_pretrained(lora_save_path)

# LoRA 저장 크기 계산
lora_size = 0
for root, dirs, files in os.walk(lora_save_path):
    for f in files:
        lora_size += os.path.getsize(os.path.join(root, f))

# 크기 비교
print(f"\n📊 저장 크기 비교")
print(f"="*60)
print(f"🔹 LoRA 어댑터:  {lora_size/1024/1024:.1f} MB")
print(f"🔹 FFT 전체 모델: {fft_size/1024/1024:.1f} MB ({fft_size/1024/1024/1024:.2f} GB)")
print(f"\n📌 FFT는 LoRA 대비 약 {fft_size/max(lora_size, 1):.0f}배 큰 저장 공간 필요")

# 정리
del lora_model_2
gc.collect()
torch.cuda.empty_cache()

# 임시 파일 정리
for path in ["./output/lora_speed_test", "./output/fft_speed_test"]:
    if os.path.exists(path):
        shutil.rmtree(path)
print("✅ 임시 파일 정리 완료")

## 7️⃣ 결과 분석 및 선택 가이드

In [ ]:
# 종합 비교표
print("="*70)
print("📊 LoRA vs FFT 종합 비교 결과")
print("="*70)
print(f"\n{'항목':<20} {'LoRA':<25} {'FFT':<25}")
print("-"*70)
print(f"{'학습 파라미터':<20} {'~1% (수십만 개)':<25} {'100% (15억 개)':<25}")
print(f"{'GPU 메모리':<20} {f'~{lora_peak_mem:.1f}GB':<25} {f'~{fft_peak_mem:.1f}GB':<25}")
print(f"{'학습 시간 (1 epoch)':<20} {f'{lora_time:.1f}초':<25} {f'{fft_time:.1f}초':<25}")
print(f"{'Training Loss':<20} {f'{lora_loss:.4f}':<25} {f'{fft_loss:.4f}':<25}")
print(f"{'저장 크기':<20} {f'{lora_size/1024/1024:.1f}MB':<25} {f'{fft_size/1024/1024:.0f}MB':<25}")
print(f"{'RTX 4060 최대 모델':<20} {'7B (QLoRA)':<25} {'1.5B':<25}")
print("-"*70)

In [ ]:
# 선택 가이드
print("\n📋 LoRA vs FFT 선택 가이드")
print("="*60)

print("""
🔹 LoRA를 선택해야 하는 경우:
   ✅ GPU 메모리가 제한적 (8GB 이하)
   ✅ 큰 모델을 사용해야 하는 경우 (7B+)
   ✅ 여러 태스크용 어댑터를 별도 관리하고 싶은 경우
   ✅ 빠른 실험/프로토타이핑이 필요한 경우
   ✅ 원본 모델을 보존하면서 커스터마이징하고 싶은 경우

🔹 FFT를 선택해야 하는 경우:
   ✅ 충분한 GPU 메모리가 있는 경우 (24GB+)
   ✅ 최고 성능이 필요한 경우
   ✅ 모델 구조를 크게 변경해야 하는 경우
   ✅ 데이터가 충분히 많은 경우
   ✅ 특정 도메인에 완전히 특화시켜야 하는 경우

📌 실무 권장사항:
   → 먼저 LoRA로 빠르게 실험하고,
   → 성능이 부족하면 FFT를 고려하세요!
""")

## 📝 정리 및 핵심 요약

### 이번 실습에서 배운 내용

| 항목 | LoRA | FFT |
|------|------|-----|
| 학습 파라미터 | ~1% (수십만 개) | 100% (15억 개) |
| GPU 메모리 | 적음 (4bit + LoRA) | 많음 (FP16 전체) |
| 학습 속도 | 빠름 | 느림 |
| 저장 크기 | ~수십 MB | ~수 GB |

### 핵심 포인트

- 🎯 **LoRA는 전체 파라미터의 ~1%만 학습**하므로 메모리와 시간이 절약됩니다
- 🎯 **FFT는 최고 성능**을 달성할 수 있지만, 자원이 많이 필요합니다
- 🎯 **RTX 4060에서 FFT는 1.5B까지만** 가능합니다
- 🎯 **실무에서는 LoRA를 먼저 시도**하는 것이 효율적입니다
- 🎯 여러 태스크에 대해 **LoRA 어댑터를 교체**하면 하나의 base 모델로 다양한 용도로 사용 가능

### 다음 단계

- ➡️ **Notebook 20**: LoRA vs FFT 실전 비교 - 동일 데이터로 학습 후 성능 비교